# ECG Anomaly Detection - MIT-BIH Database Exploration

In [ ]:
"""
Notebook: 01_explore_mit_bih.ipynb
Exploration of MIT-BIH Arrhythmia Database
"""

# 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
import pandas as pd
from pathlib import Path
import sys

# Add project root to path
sys.path.insert(0, '..')

from src.data.loader import ECGLoader
from src.data.preprocessor import ECGPreprocessor
from src.data.segmenter import ECGSegmenter

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# 2. Load MIT-BIH Data

In [ ]:
# Initialize loader
loader = ECGLoader(data_dir="../data/raw")

# Load a sample record (record 100 - normal sinus rhythm)
record_100 = loader.load_mit_bih_record(100, leads=[0, 1])

print(f"Record ID: {record_100.record_id}")
print(f"Signal shape: {record_100.signal.shape}")
print(f"Sampling rate: {record_100.sampling_rate} Hz")
print(f"Duration: {record_100.signal.shape[0] / record_100.sampling_rate:.2f} seconds")
print(f"Leads: {record_100.metadata['lead_names']}")

# 3. Visualize ECG Signals

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

time = np.arange(record_100.signal.shape[0]) / record_100.sampling_rate
duration_to_plot = 5  # seconds
samples_to_plot = int(duration_to_plot * record_100.sampling_rate)

# Lead 0 (MLII)
axes[0].plot(time[:samples_to_plot], record_100.signal[:samples_to_plot, 0], 
             'b-', linewidth=1)
axes[0].set_ylabel('Amplitude (mV)')
axes[0].set_title('Lead MLII - Normal Sinus Rhythm')
axes[0].grid(True, alpha=0.3)

# Lead 1 (V5)
axes[1].plot(time[:samples_to_plot], record_100.signal[:samples_to_plot, 1], 
             'g-', linewidth=1)
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Amplitude (mV)')
axes[1].set_title('Lead V5')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4. Analyze Annotations

In [ ]:
# Load annotations for record 100
annotation = wfdb.rdann("../data/raw/mit-bih/100", 'atr')

print(f"Number of annotations: {len(annotation.sample)}")
print(f"Unique symbols: {np.unique(annotation.symbol)}")

# Create dataframe of annotations
ann_df = pd.DataFrame({
    'sample': annotation.sample,
    'symbol': annotation.symbol,
    'time_seconds': annotation.sample / record_100.sampling_rate
})

# Map symbols to descriptions
symbol_desc = {
    1: 'Normal', 2: 'PVC', 3: 'PAC', 4: 'Ventricular Escape',
    5: 'Atrial Escape', 6: 'Nodal Escape', 7: 'Paced Beat',
    8: 'Fusion', 9: 'Unknown', 10: 'Ventricular Flutter',
    11: 'AFib', 12: 'Atrial Flutter', 13: 'Bradycardia',
    14: 'Tachycardia', 15: 'Heart Block', 16: 'Other'
}

ann_df['description'] = ann_df['symbol'].map(symbol_desc)

print("\nAnnotation distribution:")
print(ann_df['description'].value_counts())

# %%
# Visualize annotations on ECG
fig, ax = plt.subplots(figsize=(14, 6))

# Plot ECG signal
ax.plot(time[:samples_to_plot], record_100.signal[:samples_to_plot, 0], 
        'b-', linewidth=1, alpha=0.7)

# Mark annotations
for _, ann in ann_df.iterrows():
    if ann['time_seconds'] < duration_to_plot:
        y_pos = record_100.signal[int(ann['sample']), 0]
        ax.axvline(ann['time_seconds'], color='red', alpha=0.5, linewidth=0.5)
        ax.text(ann['time_seconds'], y_pos + 0.1, ann['description'], 
               fontsize=8, rotation=45, ha='right')

ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Amplitude (mV)')
ax.set_title('ECG Signal with Annotations')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 5. Explore Different Arrhythmia Types

In [ ]:
# Load records with different arrhythmias
arrhythmia_records = {
    'Normal': 100,
    'PVC (Premature Ventricular Contraction)': 119,
    'AFib (Atrial Fibrillation)': 201,
    'Bradycardia': 212,
    'Tachycardia': 208
}

fig, axes = plt.subplots(len(arrhythmia_records), 1, figsize=(14, 12))

for idx, (condition, record_num) in enumerate(arrhythmia_records.items()):
    # Load record
    record = loader.load_mit_bih_record(record_num, leads=[0])
    signal = record.signal[:int(5 * record.sampling_rate), 0]
    time = np.arange(len(signal)) / record.sampling_rate
    
    # Plot
    axes[idx].plot(time, signal, 'b-', linewidth=1)
    axes[idx].set_ylabel('Amplitude (mV)')
    axes[idx].set_title(f'{condition} - Record {record_num}')
    axes[idx].grid(True, alpha=0.3)
    
    # Highlight R-peaks (simplified detection)
    from scipy.signal import find_peaks
    peaks, _ = find_peaks(signal, distance=record.sampling_rate * 0.3)
    axes[idx].scatter(time[peaks[:10]], signal[peaks[:10]], 
                     c='red', s=30, marker='^', zorder=5)

axes[-1].set_xlabel('Time (seconds)')
plt.tight_layout()
plt.show()

# 6. Heart Rate Analysis

In [ ]:
# Calculate heart rate for each record
heart_rates = {}

for record_num in [100, 101, 103, 105, 106, 108, 109, 111, 112, 113, 114, 115, 116]:
    record = loader.load_mit_bih_record(record_num, leads=[0])
    signal = record.signal[:, 0]
    
    # Simple peak detection
    from scipy.signal import find_peaks
    peaks, _ = find_peaks(signal, distance=record.sampling_rate * 0.3)
    
    if len(peaks) > 1:
        rr_intervals = np.diff(peaks) / record.sampling_rate
        heart_rate = 60 / np.mean(rr_intervals)
        heart_rates[record_num] = heart_rate

# Display heart rates
hr_df = pd.DataFrame(list(heart_rates.items()), columns=['Record', 'Heart Rate (BPM)'])
print(hr_df)

# %%
# Plot heart rate distribution
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(hr_df['Record'].astype(str), hr_df['Heart Rate (BPM)'], 
       color='steelblue', edgecolor='black')
ax.axhline(60, color='red', linestyle='--', label='Bradycardia threshold')
ax.axhline(100, color='orange', linestyle='--', label='Tachycardia threshold')
ax.set_xlabel('Record Number')
ax.set_ylabel('Heart Rate (BPM)')
ax.set_title('Heart Rate Distribution Across MIT-BIH Records')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 7. Summary Statistics

In [ ]:
# Load multiple records to get statistics
records_to_analyze = [100, 101, 103, 105, 106, 108, 109, 111, 112, 113, 
                      114, 115, 116, 117, 118, 119, 121, 122, 123, 124]

stats = []
for record_num in records_to_analyze[:10]:  # First 10 records
    try:
        record = loader.load_mit_bih_record(record_num, leads=[0])
        signal = record.signal[:, 0]
        
        # Basic statistics
        stats.append({
            'Record': record_num,
            'Duration (min)': signal.shape[0] / record.sampling_rate / 60,
            'Mean (mV)': np.mean(signal),
            'Std (mV)': np.std(signal),
            'Min (mV)': np.min(signal),
            'Max (mV)': np.max(signal),
            'SNR (dB)': 20 * np.log10(np.std(signal) / np.std(np.diff(signal)))
        })
    except Exception as e:
        print(f"Error loading record {record_num}: {e}")

stats_df = pd.DataFrame(stats)
print(stats_df)

# %%
print("\n" + "="*60)
print("MIT-BIH Database Summary")
print("="*60)
print(f"Total records in database: 48")
print(f"Sampling frequency: 360 Hz")
print(f"Record duration: ~30 minutes each")
print(f"Total annotations: ~110,000")
print(f"Common arrhythmias: Normal, PVC, PAC, AFib, Bradycardia, Tachycardia")
print("="*60)